## SQL WITH: a running total of solar

In [1]:
import sqlite3
import pandas as pd
con = sqlite3.connect("power.db")
sql = """
WITH yearly AS (
SELECT substr(month, 1, 4) AS year,
ROUND(SUM(gwh)) AS solar
FROM generation WHERE fuel = 'Solar'
GROUP BY year)
SELECT year, solar,
SUM(solar) OVER (ORDER BY year) AS running
FROM yearly
"""
print(pd.read_sql(sql, con).tail(3))

   year     solar   running
4  2023  113409.0  381827.0
5  2024  133795.0  515622.0
6  2025   97156.0  612778.0


## SQL RANK: the leading solar states

In [2]:
sql = """
WITH t AS (
SELECT state, ROUND(SUM(gwh)) AS solar
FROM generation WHERE fuel = 'Solar'
AND month LIKE '2024%'
GROUP BY state)
SELECT state, solar,
RANK() OVER (ORDER BY solar DESC) AS rnk
FROM t LIMIT 3
"""
print(pd.read_sql(sql, con))

  state    solar  rnk
0    RJ  46040.0    1
1    GJ  17374.0    2
2    KA  15398.0    3


## SQL LAG: month-over-month solar in 2025

In [3]:
sql = """
WITH m AS (
SELECT month, ROUND(SUM(gwh)) AS solar
FROM generation WHERE fuel = 'Solar'
AND month >= '2025-04'
GROUP BY month)
SELECT month, solar,
LAG(solar) OVER (ORDER BY month) AS before
FROM m
"""
print(pd.read_sql(sql, con))

     month    solar   before
0  2025-04  15812.0      NaN
1  2025-05  14305.0  15812.0
2  2025-06  12927.0  14305.0
3  2025-07  12101.0  12927.0
